# 17 LlamaTelemetry + Weights & Biases (Kaggle)

Combine llama.cpp inference, OTLP telemetry, and W&B experiment tracking.

**What you will learn:**
- Load W&B and Grafana secrets from Kaggle
- Initialize a W&B run
- Start llama-server from a preset
- Run instrumented inference and log metrics to W&B
- Log server health data

**Requirements:** Kaggle T4 x2. Kaggle Secrets: `WANDB_API_KEY` (required),
`GRAFANA_OTLP_ENDPOINT` / `GRAFANA_OTLP_HEADERS` (optional).

## 1) Install

In [ ]:
!pip -q install git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.0
!pip -q install wandb

## 2) Load Kaggle Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    WANDB_API_KEY = secrets.get_secret('WANDB_API_KEY')
    GRAFANA_OTLP_ENDPOINT = secrets.get_secret('GRAFANA_OTLP_ENDPOINT')
    GRAFANA_OTLP_HEADERS = secrets.get_secret('GRAFANA_OTLP_HEADERS')
except Exception:
    WANDB_API_KEY = os.environ.get('WANDB_API_KEY')
    GRAFANA_OTLP_ENDPOINT = None
    GRAFANA_OTLP_HEADERS = None

print(f"WANDB_API_KEY set: {bool(WANDB_API_KEY)}")
print(f"OTLP endpoint set: {bool(GRAFANA_OTLP_ENDPOINT)}")

## 3) Initialize W&B

In [ ]:
if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY

import wandb

run = wandb.init(
    project='llamatelemetry',
    name='kaggle-llama.cpp-e2',
    reinit=True,
    config={
        'sdk_version': '0.1.0',
        'preset': 'KAGGLE_DUAL_T4',
    },
)
print(f"W&B run: {run.name}")

## 4) Start llama-server

In [ ]:
from llamatelemetry.kaggle import ServerPreset
from llamatelemetry.kaggle.pipeline import start_server_from_preset

model_path = '/kaggle/input/your-model/model.gguf'
manager = start_server_from_preset(model_path, ServerPreset.KAGGLE_DUAL_T4)
print(f"Server healthy: {manager.check_server_health()}")

## 5) Setup telemetry + instrumented client

In [ ]:
from llamatelemetry.kaggle.pipeline import (
    KagglePipelineConfig,
    load_grafana_otlp_env_from_kaggle,
    setup_otel_and_client,
)

load_grafana_otlp_env_from_kaggle()

cfg = KagglePipelineConfig(
    enable_graphistry=False,
    enable_llama_metrics=True,
)
ctx = setup_otel_and_client('http://127.0.0.1:8080', cfg)
client = ctx['client']
print(f"Pipeline keys: {list(ctx.keys())}")

## 6) Run inference + log to W&B

In [ ]:
queries = [
    'Explain llama.cpp in one paragraph.',
    'What is CUDA?',
    'How does quantization reduce model size?',
]

for i, query in enumerate(queries):
    resp = client.chat_completions({
        'model': 'local-gguf',
        'messages': [{'role': 'user', 'content': query}],
        'max_tokens': 64,
    })
    text = resp.choices[0].message.content
    print(f"[{i+1}] {text[:100]}...")

    wandb.log({
        'query': query,
        'output_preview': text[:200],
        'response_words': len(text.split()),
        'step': i,
    })

## 7) Log server health to W&B

In [ ]:
health = manager.get_health()
props = manager.get_props()

wandb.log({
    'server_health': str(health),
    'model_path': props.get('model_path') if isinstance(props, dict) else None,
})
print(f"Health: {health}")

## 8) Finish W&B run + cleanup

In [ ]:
wandb.finish()
manager.stop_server()
print("Done.")